In [ ]:
import pandas as pd
from main import SatCLIPLightningModule
import location_encoder as LE
from temporal_encoding import Fourier, Direct
from tqdm import tqdm

In [ ]:
# Load SatCLIP model
ckpt_path = '/home/leca5365/Documents/satclip/models/satclip-fixed-fourier/checkpoints/best.ckpt'
lightning_model = SatCLIPLightningModule.load_from_checkpoint(ckpt_path)

lightning_model.eval()
spatiotemporal_enc = lightning_model.model.location
visual_enc = lightning_model.model.visual

using pretrained moco vit16


In [20]:
# Load the GHCN dataset
ghcn_df = pd.read_csv("../notebooks/ghcn_2021_2023_temperature.csv")

KeyboardInterrupt: 

In [ ]:
ghcn_df.head()

,ID,Date,Element,Value,MFlag,QFlag,SFlag,OBS-TIME,Latitude,Longitude,Elevation,Name
0,USC00171430,20210101,TMAX,-6,NaN,NaN,7,700.0,46.4925,-69.2864,284.7,CHURCHILL DAM
1,USC00171430,20210101,TMIN,-128,NaN,NaN,7,700.0,46.4925,-69.2864,284.7,CHURCHILL DAM
2,USC00171628,20210101,TMAX,39,NaN,NaN,7,700.0,44.9197,-69.2417,90.5,CORINNA
3,USC00171628,20210101,TMIN,-111,NaN,NaN,7,700.0,44.9197,-69.2417,90.5,CORINNA
4,USC00171867,20210101,TMAX,28,NaN,NaN,7,700.0,44.6789,-68.6356,141.1,DEDHAM


In [ ]:
# create MLP head for temperature prediction
import torch
import torch.nn as nn

class TemperaturePredictor(nn.Module):
    def __init__(self, location_encoder, embed_dim=256, hidden_dim=512):
        super(TemperaturePredictor, self).__init__()
        self.location_encoder = location_encoder
        self.fc1 = nn.Linear(embed_dim, hidden_dim)
        self.fc2 = nn.Linear(hidden_dim, 1)  # Predict temperature

    def forward(self, coord):
        loc_emb = self.location_encoder(coord)
        x = torch.relu(self.fc1(loc_emb))
        temp_pred = self.fc2(x)
        return temp_pred



In [60]:
# Create dataloader
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

from torch.utils.data import DataLoader, TensorDataset, random_split

# GHCN to day-of-year
dataset_df = ghcn_df[(ghcn_df["Element"]=="TMAX") & (ghcn_df["QFlag"].isna())]
dataset_df['Date'] = pd.to_datetime(dataset_df['Date'], format="%Y%m%d")
dataset_df['DayOfYear'] = dataset_df['Date'].dt.dayofyear

# Prepare dataset
coords = torch.tensor(dataset_df[['Longitude', 'Latitude', 'DayOfYear']].values, dtype=torch.float32).double().to(device)
temperatures = torch.tensor(dataset_df['Value'].values, dtype=torch.float32).unsqueeze(1).double().to(device)

generator = torch.Generator().manual_seed(42)

dataset = TensorDataset(coords, temperatures)

train, val = random_split(TensorDataset(coords, temperatures), [int(0.8*len(coords)), len(coords) - int(0.8*len(coords))], generator=generator)

dataloader = DataLoader(train, batch_size=262144, shuffle=True)
dataloader_val = DataLoader(val, batch_size=262144, shuffle=False)

/tmp/ipykernel_351927/4264372034.py:8: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  dataset_df['Date'] = pd.to_datetime(dataset_df['Date'], format="%Y%m%d")
/tmp/ipykernel_351927/4264372034.py:9: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  dataset_df['DayOfYear'] = dataset_df['Date'].dt.dayofyear


In [30]:
# Fine-tune the predictor head
from tqdm import tqdm

tp = TemperaturePredictor(spatiotemporal_enc)

for param in tp.location_encoder.parameters():
    param.requires_grad = False

opt = torch.optim.AdamW(tp.parameters(), lr=1e-3, weight_decay=False)
criterion = nn.MSELoss()

tp.to(device)
tp.double()

num_epochs = 10
for epoch in range(num_epochs):
    tp.train()
    total_loss = 0

    for batch in tqdm(dataloader):
        coords_batch, temp_batch = batch
        # coords_batch, temp_batch = coords_batch.to(device), temp_batch.to(device)
        opt.zero_grad()
        temp_pred = tp(coords_batch)
        loss = criterion(temp_pred, temp_batch)
        loss.backward()
        opt.step()
        total_loss += loss.item()

    avg_loss = total_loss / len(dataloader)
    print(f"Epoch {epoch+1}/{num_epochs}, Loss: {avg_loss:.4f}")

  0%|          | 0/41 [00:00<?, ?it/s]

100%|██████████| 41/41 [01:30<00:00,  2.20s/it]


Epoch 1/10, Loss: 39106.6288


100%|██████████| 41/41 [01:30<00:00,  2.20s/it]


Epoch 2/10, Loss: 23184.7080


...

Epoch 8/10, Loss: 12656.1868


100%|██████████| 41/41 [01:30<00:00,  2.20s/it]


Epoch 9/10, Loss: 12442.8892


100%|██████████| 41/41 [01:30<00:00,  2.21s/it]

Epoch 10/10, Loss: 12285.2769


In [31]:
total_loss = 0
for batch in tqdm(dataloader_val):
    coords_batch, temp_batch = batch

    with torch.no_grad():
        temp_pred = tp(coords_batch)
        loss = criterion(temp_pred, temp_batch)
        total_loss += loss.item()

print(f"Validation Loss: {total_loss/len(dataloader_val):.4f}")

100%|██████████| 11/11 [00:20<00:00,  1.86s/it]

Validation Loss: 12210.2172


In [36]:
12210.2172*0.1 / (len(dataset)*0.2)

0.00046189261539525384

In [59]:
ghcn_df[(ghcn_df["Element"]=="TMAX") & (ghcn_df["QFlag"].isna())]["Value"].max()

np.int64(544)

In [46]:
test_coord = torch.tensor([-69.2864,46.4925,3]).to(device).double().unsqueeze(0)
tp(test_coord)

tensor([[96.5283]], device='cuda:0', dtype=torch.float64,
       grad_fn=<AddmmBackward0>)